# batch2_A: the questions' prompts and the complete rerun inputs

Step 2 of "How batch2_A's sheets are made" (`prereg_batch2_A.md`), on Colab's CPU (no GPU needed). It unpacks the code
and the scenes' geometry (16 frames, cameras, depth, `adapter_report.json`, made locally with `adapt_vggt.py`), takes
each question's SAM 3 prompts with `prompt_object_extractor.py` (spaCy 3.8 with `en_core_web_md` 3.8.0 and NLTK
WordNet, the versions `hpc/setup_env.sh` installs), writes each sheet folder's `question.json` and `index.csv`
(`batch_inputs.py questions`), and saves the whole folder to Drive as `rerun_inputs_batch2_A.zip`, which
`rerun_detection_batch2_A.ipynb` then reads. It prints counts only, no answers.

**Files on Google Drive**, in `MyDrive/mvp/`: `mvp-3d-vlm-rerun-batch2.zip` and `rerun_inputs_batch2_A_geometry.zip`.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/mvp"
CODE_ZIP = f"{DRIVE_DIR}/mvp-3d-vlm-rerun-batch2.zip"
GEOMETRY_ZIP = f"{DRIVE_DIR}/rerun_inputs_batch2_A_geometry.zip"
OUT_ZIP = f"{DRIVE_DIR}/rerun_inputs_batch2_A.zip"

import shutil, subprocess, sys, zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
for z in (CODE_ZIP, GEOMETRY_ZIP):
    assert Path(z).is_file(), f"missing {z}: upload it to {DRIVE_DIR}"
for d in ("/content/mvp-3d-vlm", "/content/rerun_inputs"):
    shutil.rmtree(d, ignore_errors=True)
zipfile.ZipFile(CODE_ZIP).extractall("/content")
zipfile.ZipFile(GEOMETRY_ZIP).extractall("/content")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "spacy>=3.8,<3.9", "nltk",
                "en_core_web_md @ https://github.com/explosion/spacy-models/releases/download/en_core_web_md-3.8.0/"
                "en_core_web_md-3.8.0-py3-none-any.whl"], check=True)
print("unpacked:", len(list(Path("/content/rerun_inputs").glob("*__*"))), "scenes")

In [ ]:
import csv, hashlib, json
from collections import Counter

E = "/content/mvp-3d-vlm/experiments/2026-11-18-scene-sheet-formats"
r = subprocess.run([sys.executable, f"{E}/batch_inputs.py", "questions", "--scenes", f"{E}/batch2_A_scenes.txt",
                    "--out", "/content/rerun_inputs"], capture_output=True, text=True)
print(r.stdout[-1500:])
assert r.returncode == 0, r.stderr[-3000:]
INPUTS = Path("/content/rerun_inputs")
rows = list(csv.DictReader(open(INPUTS / "index.csv")))
modes = Counter(json.loads((INPUTS / row["pipeline_sheet"] / "question.json").read_text())["prompt_mode"] for row in rows)
assert len(rows) == 798, len(rows)
print(f"{len(rows)} questions in {len({row['scene'] for row in rows})} scenes; prompt modes {dict(modes)}; "
      f"without a prompt: {sum(not row['prompts'] for row in rows)}; "
      f"distinct prompts per scene: {sorted(len({p for row in rows if row['scene'] == s for p in row['prompts'].split(', ') if p}) for s in {row['scene'] for row in rows})}")
local = shutil.make_archive("/content/rerun_inputs_batch2_A", "zip", root_dir="/content", base_dir="rerun_inputs")
shutil.copy(local, OUT_ZIP)
print(f"saved {OUT_ZIP} ({Path(OUT_ZIP).stat().st_size / 2**20:.0f} MB), sha256 {hashlib.sha256(Path(OUT_ZIP).read_bytes()).hexdigest()}")